# DC扫描API示例 - Id-Vd特性测量

使用新的 `DCSweepAPI` 执行 Id-Vd 扫描（输出特性）。

**相比旧的09 notebook：**
- ✅ 更简洁的代码
- ✅ 统一的API接口
- ✅ 自动配置管理和错误处理
- ✅ 一致的数据导出格式

In [ ]:
# 导入
import yaml
from pathlib import Path
from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.dc import DCSweepAPI

In [ ]:
# 从配置文件加载（可选方式）
with open("configs/instruments.yaml", 'r', encoding='utf-8') as f:
    inst_cfg = yaml.safe_load(f)['b1500']

with open("configs/channel_map.yaml", 'r', encoding='utf-8') as f:
    roles = yaml.safe_load(f)['current_device']['role_map']

# 构建VISA配置
visa_cfg = VisaConfig(
    resource=inst_cfg['resource'],
    timeout_ms=inst_cfg['timeout_ms'],
    write_termination=inst_cfg['write_termination'],
    read_termination=inst_cfg['read_termination'],
    send_end=inst_cfg['send_end']
)

CH_G = roles['G']
CH_D = roles['D']
CH_S = roles['S']

print(f"通道映射: G={CH_G}, D={CH_D}, S={CH_S}")

In [ ]:
# 扫描参数
vg_points = [0.0, -0.4, -0.8]    # 选定的Vg点
vd_points = [0.0, 0.05, 0.10]    # Vd扫描范围
vs_fixed = 0.0                    # 固定Vs

In [ ]:
# 执行Id-Vd扫描
with VisaSession(visa_cfg) as session:
    # 初始化API
    dc_api = DCSweepAPI(
        session=session,
        ch_g=CH_G,
        ch_d=CH_D,
        ch_s=CH_S
    )
    
    # 运行Id-Vd扫描
    result = dc_api.run_idvd_sweep(
        vg_points=vg_points,
        vd_points=vd_points,
        vs_fixed=vs_fixed,
        auto_export=True,
        verbose=True
    )

# 提取结果
df = result['df']
run_dir = result['run_dir']
qc_df = result['qc_df']

In [ ]:
# 查看测量数据
df[['vg_set', 'vd_set', 'id_A', 'ig_A', 'status']]

In [ ]:
# 查看QC报告
qc_df

In [ ]:
# 绘制Id-Vd曲线（不同Vg）
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

for vg in vg_points:
    df_vg = df[df['vg_set'] == vg]
    plt.plot(df_vg['vd_set'], df_vg['id_A'].abs(), 'o-', label=f'Vg={vg}V')

plt.xlabel('Vd (V)')
plt.ylabel('|Id| (A)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.title(f'Id-Vd Output Characteristics (Vs={vs_fixed}V)')
plt.tight_layout()
plt.show()

In [ ]:
# 打印保存路径
print(f"\n数据已保存到: {run_dir}")
print(f"  - 数据文件: {result['data_paths']['csv']}")
print(f"  - QC报告: {run_dir / 'qc.csv'}")